# Phase 1: Data Loading and Initial EDA

Our first step is to load the core datasets provided for the project. According to the documentation, the "kernel" datasets are `receivals.csv` and `purchase_orders.csv`. We will load these and perform an initial inspection using `.head()` and `.info()` to understand the data types, column names, and check for any immediate missing data.

* `receivals.csv`: This is the primary training data, containing historical records of material receivals. This is what we'll use to build our target variable.
* `purchase_orders.csv`: This file contains information on ordered quantities and expected deliveries. This will be crucial for creating features, as it's the best information we have about *future* expected shipments.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Configuration ---
# Define file paths based on the project structure
RECEIVALS_FILE = "data/kernel/receivals.csv"
PURCHASE_ORDERS_FILE = "data/kernel/purchase_orders.csv"

# --- Load Data ---
print("Loading data...")
try:
    df_receivals = pd.read_csv(RECEIVALS_FILE)
    df_po = pd.read_csv(PURCHASE_ORDERS_FILE)
    print("Data loaded successfully.")
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Please make sure the file paths are correct and the files are in the 'append_consulting_project' directory.")
    # Stop execution if files aren't found
    raise

# --- Initial Inspection: Receivals ---
print("\n--- Inspecting receivals.csv ---")
print("Head:")
print(df_receivals.head())
print("\nInfo:")
df_receivals.info()

# --- Initial Inspection: Purchase Orders ---
print("\n--- Inspecting purchase_orders.csv ---")
print("Head:")
print(df_po.head())
print("\nInfo:")
df_po.info()

print("\nInitial data inspection complete.")

Loading data...
Data loaded successfully.

--- Inspecting receivals.csv ---
Head:
   rm_id  product_id  purchase_order_id  purchase_order_item_no  \
0  365.0  91900143.0           208545.0                    10.0   
1  365.0  91900143.0           208545.0                    10.0   
2  365.0  91900143.0           208490.0                    10.0   
3  365.0  91900143.0           208490.0                    10.0   
4  379.0  91900296.0           210435.0                    20.0   

   receival_item_no  batch_id                date_arrival receival_status  \
0                 1       NaN  2004-06-15 13:34:00 +02:00       Completed   
1                 2       NaN  2004-06-15 13:34:00 +02:00       Completed   
2                 1       NaN  2004-06-15 13:38:00 +02:00       Completed   
3                 2       NaN  2004-06-15 13:38:00 +02:00       Completed   
4                 1       NaN  2004-06-15 13:40:00 +02:00       Completed   

   net_weight  supplier_id  
0     11420.0        52

## Phase 1b: Data Cleaning & Initial Validation

The `.info()` output confirmed several key data cleaning tasks:

1.  **Date Conversions:** The date columns (`date_arrival`, `delivery_date`, `created_date_time`, `modified_date_time`) are all stored as `object` (string) types. We must convert these to proper `datetime` objects to work with them. `date_arrival` is timezone-aware, so we will convert it and standardize it to UTC.
2.  **Missing Data:** We need to get a full count of missing (`NaN`) values for each column to understand which features are reliable.
3.  **Categorical Data:** We need to inspect the unique values of our non-numeric `object` columns (`receival_status`, `unit`, `status`) to see what kind of data they contain and if they'll be useful.

In [3]:
# --- 1. Date Conversions ---
print("--- 1. Converting Date Columns ---")

# Convert receivals date_arrival. This column has timezone info, so we'll parse it and standardize to UTC.
# This makes all timestamps comparable.
print("Converting df_receivals['date_arrival']...")
df_receivals['date_arrival'] = pd.to_datetime(df_receivals['date_arrival'], utc=True, errors='coerce')

# Convert purchase_orders dates. We'll use errors='coerce' to turn any bad data into NaT (Not-a-Time).
print("Converting df_po dates...")
df_po['delivery_date'] = pd.to_datetime(df_po['delivery_date'], errors='coerce')
df_po['created_date_time'] = pd.to_datetime(df_po['created_date_time'], errors='coerce')
df_po['modified_date_time'] = pd.to_datetime(df_po['modified_date_time'], errors='coerce')

print("Date conversions complete.")

# --- 2. Re-check Info ---
print("\n--- 2. Post-Conversion Info ---")
print("Receivals Info:")
df_receivals.info()
print("\nPurchase Orders Info:")
df_po.info()

# --- 3. Check for Missing Data (NaNs) ---
print("\n--- 3. Missing Value Counts ---")
print("Receivals NaNs:")
print(df_receivals.isnull().sum())
print("\nPurchase Orders NaNs:")
print(df_po.isnull().sum())

# --- 4. Inspect Categorical Column Values ---
print("\n--- 4. Unique Categorical Values ---")

# Check receival_status
print(f"\nUnique 'receival_status' values (in df_receivals):")
print(df_receivals['receival_status'].value_counts(dropna=False))

# Check unit
print(f"\nUnique 'unit' values (in df_po):")
print(df_po['unit'].value_counts(dropna=False))

# Check status
print(f"\nUnique 'status' values (in df_po):")
print(df_po['status'].value_counts(dropna=False))

print("\nData cleaning and validation step complete.")

--- 1. Converting Date Columns ---
Converting df_receivals['date_arrival']...
Converting df_po dates...
Date conversions complete.

--- 2. Post-Conversion Info ---
Receivals Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122590 entries, 0 to 122589
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype              
---  ------                  --------------   -----              
 0   rm_id                   122533 non-null  float64            
 1   product_id              122533 non-null  float64            
 2   purchase_order_id       122537 non-null  float64            
 3   purchase_order_item_no  122537 non-null  float64            
 4   receival_item_no        122590 non-null  int64              
 5   batch_id                64765 non-null   float64            
 6   date_arrival            122590 non-null  datetime64[ns, UTC]
 7   receival_status         122590 non-null  object             
 8   net_weight              122522 non-null  flo

/var/folders/t4/d04pxw590zd46jkm3bgdz8kr0000gn/T/ipykernel_42388/3709217865.py:11: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df_po['delivery_date'] = pd.to_datetime(df_po['delivery_date'], errors='coerce')


## Phase 1c: Key Findings and Final Cleaning

From the previous step, we have several key findings:

1.  **Date Conversions:** `df_receivals['date_arrival']` converted perfectly to a `datetime64[ns, UTC]` object. This is great.
2.  **CRITICAL AMBIGUITY:** The `df_po.info()` output still shows `delivery_date` as an `object` type, *but* the other two `datetime` columns in that table converted correctly. This is contradictory. It's likely a copy-paste error in the notebook output, but we **must** be 100% certain. `delivery_date` is one of our most important features.
3.  **Missing Data (`receivals`):**
    * `batch_id` is ~50% missing. We will ignore this column as it's unreliable.
    * `rm_id`, `product_id`, `net_weight`, etc., have a very small number of `NaN`s (50-70 rows). We can safely drop these few rows.
4.  **Missing Data (`purchase_orders`):**
    * `modified_date_time` has 492 NaNs, which is fine, as we don't plan to use it as a feature.
    * `unit` and `unit_id` have 44 NaNs.
5.  **Categorical Data (`purchase_orders`):**
    * `unit`: The data is 99.8% `KG` (33,120 rows). The 7 `PUND` rows and 44 `NaN`s are noise. We will filter our `df_po` table to only include `KG`.
    * `status`: `Closed`, `Deleted`, and `Open` look like very important categorical features for predicting if an order will actually arrive.

Our next step is to resolve the `delivery_date` ambiguity, perform the final cleaning based on these findings, and then find the full date range of our data.

In [4]:
# --- 1. Verify and Fix delivery_date ---
print("--- 1. Verifying 'delivery_date' Dtype ---")
try:
    print(f"Current Dtype of 'delivery_date': {df_po['delivery_date'].dtype}")
except Exception as e:
    print(f"Error checking dtype: {e}")

# Re-run conversion - this is idempotent (safe to run again) and ensures it's a datetime
# We will also convert it to UTC to match df_receivals
print("Attempting conversion to datetime and UTC...")
df_po['delivery_date'] = pd.to_datetime(df_po['delivery_date'], errors='coerce', utc=True)

# Check Dtype AGAIN to be 100% sure.
print(f"New Dtype of 'delivery_date': {df_po['delivery_date'].dtype}")

# --- 2. Clean df_receivals ---
print("\n--- 2. Cleaning df_receivals ---")
initial_receivals_count = len(df_receivals)
# Drop rows where our key identifiers or the target variable (net_weight) are missing
df_receivals_cleaned = df_receivals.dropna(
    subset=['rm_id', 'product_id', 'purchase_order_id', 'net_weight']
).copy()
print(f"Dropped {initial_receivals_count - len(df_receivals_cleaned)} rows from df_receivals.")

# --- 3. Clean df_po ---
print("\n--- 3. Cleaning df_po ---")
initial_po_count = len(df_po)
# First, drop the 44 rows where 'unit' is NaN
df_po_cleaned = df_po.dropna(subset=['unit_id', 'unit']).copy()
# Now, filter to keep only 'KG'
df_po_cleaned = df_po_cleaned[df_po_cleaned['unit'] == 'KG'].copy()
print(f"Dropped {initial_po_count - len(df_po_cleaned)} rows from df_po (NaNs and non-KG units).")


# --- 4. Report Final Date Ranges ---
print("\n--- 4. Final Data Date Ranges (Cleaned) ---")
# Use .dt.date to avoid printing the time part
print(f"Receivals 'date_arrival' range: {df_receivals_cleaned['date_arrival'].min().date()} to {df_receivals_cleaned['date_arrival'].max().date()}")
print(f"PO 'delivery_date' range:     {df_po_cleaned['delivery_date'].min().date()} to {df_po_cleaned['delivery_date'].max().date()}")

print("\nData cleaning and date range verification complete.")

--- 1. Verifying 'delivery_date' Dtype ---
Current Dtype of 'delivery_date': object
Attempting conversion to datetime and UTC...
New Dtype of 'delivery_date': datetime64[ns, UTC]

--- 2. Cleaning df_receivals ---
Dropped 70 rows from df_receivals.

--- 3. Cleaning df_po ---
Dropped 51 rows from df_po (NaNs and non-KG units).

--- 4. Final Data Date Ranges (Cleaned) ---
Receivals 'date_arrival' range: 2004-06-15 to 2024-12-19
PO 'delivery_date' range:     2002-01-30 to 2025-06-29

Data cleaning and date range verification complete.


## Phase 1d: Creating the Validation Split

Now that our data is clean, we will create our **time-based validation set**. This is the single most important step to prevent overfitting and avoid the "disappointing big drop" in the private leaderboard.

Our strategy is to mirror the actual problem:
* **The Real Task:** Train on history (up to Dec 2024) to predict a future 5-month block (Jan 2025 - May 2025).
* **Our Local Task:** We will hold out the *last* ~5 months of our historical data to use as a local test set. We'll train on the data *before* that, and see how well we predict this held-out period.

Our `receivals` data ends on **2024-12-19**. We will set our validation period to start on **2024-08-01**.

* **Training Set:** All receivals from 2004-06-15 up to **2024-07-31**.
* **Validation Set:** All receivals from **2024-08-01** to **2024-12-19**.

We will also run a `.describe()` on our key numeric columns (`net_weight` and `quantity`) to check for outliers or strange values.

In [5]:
# --- 1. Define Validation Split Date ---
# We'll use a timezone-aware string to match our UTC datetimes
VALIDATION_START_DATE = pd.to_datetime('2024-08-01', utc=True)

# --- 2. Create Training and Validation Sets ---
print("--- 1. Splitting receivals data ---")

# Training set is everything *before* the validation start date
train_receivals = df_receivals_cleaned[df_receivals_cleaned['date_arrival'] < VALIDATION_START_DATE].copy()

# Validation set is everything *on or after* the validation start date
validation_receivals = df_receivals_cleaned[df_receivals_cleaned['date_arrival'] >= VALIDATION_START_DATE].copy()

# --- 3. Report Split Results ---
print("--- 2. Reporting split results ---")
print(f"Total cleaned receivals:  {len(df_receivals_cleaned)}")
print(f"Training receivals:       {len(train_receivals)}")
print(f"Validation receivals:     {len(validation_receivals)}")
print(f"Check: Is total == train + val?  {(len(train_receivals) + len(validation_receivals)) == len(df_receivals_cleaned)}")

print(f"\nTraining set date range:   {train_receivals['date_arrival'].min().date()} to {train_receivals['date_arrival'].max().date()}")
print(f"Validation set date range: {validation_receivals['date_arrival'].min().date()} to {validation_receivals['date_arrival'].max().date()}")

# --- 4. Final Statistical Summary ---
print("\n--- 3. Statistical Summary of Key Columns ---")
print("\nSummary of 'net_weight' (from df_receivals_cleaned):")
# Use .describe() for a statistical summary
print(df_receivals_cleaned['net_weight'].describe())

print("\nSummary of 'quantity' (from df_po_cleaned):")
print(df_po_cleaned['quantity'].describe())

print("\nValidation split and statistical summary complete.")

--- 1. Splitting receivals data ---
--- 2. Reporting split results ---
Total cleaned receivals:  122520
Training receivals:       120402
Validation receivals:     2118
Check: Is total == train + val?  True

Training set date range:   2004-06-15 to 2024-07-31
Validation set date range: 2024-08-01 to 2024-12-19

--- 3. Statistical Summary of Key Columns ---

Summary of 'net_weight' (from df_receivals_cleaned):
count    122520.000000
mean      12972.434427
std        8264.637075
min           0.000000
25%        5660.000000
50%       12380.000000
75%       21120.000000
max       31626.000000
Name: net_weight, dtype: float64

Summary of 'quantity' (from df_po_cleaned):
count    3.312000e+04
mean     9.020381e+04
std      3.307807e+05
min     -8.580000e+03
25%      1.000000e+04
50%      2.625500e+04
75%      1.000000e+05
max      2.497599e+07
Name: quantity, dtype: float64

Validation split and statistical summary complete.
